# FMP ETFs

Read-first examples for FMP-backed ETF prices, metadata, and composition data.

In [ ]:
from __future__ import annotations

from dataclasses import asdict

import pandas as pd
from openbb import obb

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

ETF_SYMBOL = "SPY"

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available ETF Sections

In [ ]:
etf_sections = ["etf_prices", "etf_profile", *FMP_HISTORICAL_ETF_SECTIONS]
etf_route_libraries = {
    "etf_prices": ETF_PRICES_LIBRARY,
    "etf_profile": "etf_profile",
    **{section: fundamental_library(section) for section in FMP_HISTORICAL_ETF_SECTIONS},
}
route_table(etf_route_libraries)

## Local Storage Coverage

In [ ]:
state_table(ETF_SYMBOL, etf_sections)

## Prices and Profile

In [ ]:
if RUN_REFRESH:
    warehouse.etf.refresh_prices(ETF_SYMBOL, providers=[PROVIDER])
    warehouse.etf.refresh_profile(ETF_SYMBOL, provider=PROVIDER)

preview(warehouse.etf.read_prices(ETF_SYMBOL, provider=PROVIDER))

In [ ]:
profile = warehouse.etf.read_profile(ETF_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

## Holdings and Exposures

In [ ]:
if RUN_REFRESH:
    warehouse.etf.refresh_fundamentals(
        ETF_SYMBOL,
        sections=["etf_holdings", "etf_sectors", "etf_countries", "etf_equity_exposure", "etf_price_performance"],
        providers=[PROVIDER],
    )

{
    "holdings": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_holdings", provider=PROVIDER)),
    "sectors": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_sectors", provider=PROVIDER)),
    "countries": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_countries", provider=PROVIDER)),
    "equity_exposure": preview(warehouse.etf.read_fundamentals(ETF_SYMBOL, section="etf_equity_exposure", provider=PROVIDER)),
}

## Direct FMP/OpenBB Route Check

In [ ]:
pd.DataFrame([
    run_openbb_route("equity.price.historical", lambda: obb.equity.price.historical(symbol=ETF_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("etf.historical", lambda: obb.etf.historical(symbol=ETF_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("equity.profile", lambda: obb.equity.profile(symbol=ETF_SYMBOL, provider=PROVIDER)),
    run_openbb_route("etf.info", lambda: obb.etf.info(symbol=ETF_SYMBOL, provider=PROVIDER)),
])